In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
pip install flwr

  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
Using cached packaging-25.0-py3-none-any.whl (66 kB)
Using cached protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl (323 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.34.1
    Uninstalling protobuf-7.34.1:
      Successfully uninstalled protobuf-7.34.1
  Attempting uninstall: packaging
    Found existing installation: packaging 26.1
    Uninstalling packaging-26.1:
      Successfully uninstalled packaging-26.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have prot

In [2]:
"""
Federated Learning — Multi Disease Detection Project
=====================================================
Framework : Flower (flwr)
Strategy  : FedAvg (with FedProx option commented in)
Model     : TB_CNN  (lightest model, ideal for FL simulation)
Clients   : 3 simulated clients, each with a partition of TB data
Run       : python ML/src/federated_tb.py
"""

import flwr as fl
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
import numpy as np
from collections import OrderedDict
import matplotlib.pyplot as plt
import os

# ─────────────────────────────────────────────
# 0. CONFIG
# ─────────────────────────────────────────────
DATA_DIR        = "/content/drive/MyDrive/Multiple Disease Detection Project/Datasets/tb merged"         # single root — NO pre-split needed
NUM_CLIENTS     = 3
NUM_ROUNDS      = 10
LOCAL_EPOCHS    = 3
BATCH_SIZE      = 32
LR              = 1e-3
TRAIN_RATIO     = 0.80                  # 80 % train
VAL_RATIO       = 0.10                  # 10 % validation
# remaining 10 % goes to test (unused here, but kept for completeness)
MODEL_SAVE_PATH = "ML/Models/tb_cnn_federated.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [3]:
class TB_CNN(nn.Module):
    def __init__(self):
        super(TB_CNN, self).__init__()
        self.conv1   = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.pool    = nn.MaxPool2d(4, 4)
        self.fc1     = nn.Linear(8 * 56 * 56, 32)
        self.dropout = nn.Dropout(0.5)
        self.fc2     = nn.Linear(32, 2)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = x.view(-1, 8 * 56 * 56)
        x = self.dropout(x)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

In [4]:
# ─────────────────────────────────────────────
# 2. DATA — load once, auto-split, then partition
#    for federated clients
# ─────────────────────────────────────────────

# --- transforms (augmentation only for training) ---
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])


class TransformSubset(torch.utils.data.Dataset):
    """Wraps a Subset so we can apply a *different* transform than
    the one baked into the original ImageFolder."""
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]       # PIL image (because base has no transform)
        if self.transform:
            img = self.transform(img)
        return img, label


# Load FULL dataset WITHOUT transforms (PIL images)
# so that each split can apply its own transforms later.
full_dataset = ImageFolder(DATA_DIR, transform=None)
print(f"Classes detected : {full_dataset.classes}")      # e.g. ['Normal', 'TB']
print(f"Total images     : {len(full_dataset)}")

# --- split into train / val / test ---
from torch.utils.data import random_split

total      = len(full_dataset)
train_size = int(TRAIN_RATIO * total)
val_size   = int(VAL_RATIO * total)
test_size  = total - train_size - val_size      # whatever remains

train_subset, val_subset, test_subset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)     # reproducible split
)
print(f"Split → train: {train_size}  |  val: {val_size}  |  test: {test_size}")

# Wrap subsets with the correct transforms
train_dataset = TransformSubset(train_subset, train_transform)
val_dataset   = TransformSubset(val_subset, val_transform)
test_dataset  = TransformSubset(test_subset, val_transform)

# --- partition training data across federated clients (IID) ---
indices        = np.random.permutation(len(train_dataset))
partition_size = len(train_dataset) // NUM_CLIENTS
partitions     = [
    indices[i * partition_size: (i + 1) * partition_size]
    for i in range(NUM_CLIENTS)
]

def get_client_loader(client_id: int):
    subset = Subset(train_dataset, partitions[client_id])
    return DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

val_loader  = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


Classes detected : ['normal', 'tb diagnosed']
Total images     : 7208
Split → train: 5766  |  val: 720  |  test: 722


In [5]:
# # These two functions are the bridge between Flower (the federated framework) and PyTorch.
# # Flower communicates model weights as NumPy arrays, but PyTorch uses tensors.
# # These helpers translate between the two formats.
# #What it does step-by-step:

# model.state_dict() — returns an OrderedDict of all learnable weights & biases, e.g.:
# conv1.weight, conv1.bias, fc1.weight, fc1.bias, fc2.weight, fc2.bias
# .values() — grabs just the tensor values (ignoring the key names)
# .cpu() — moves each tensor from GPU back to CPU (needed for NumPy conversion)
# .numpy() — converts each PyTorch tensor → NumPy array
# When is it called? After a client finishes local training, this sends the updated weights back to the Flower server for aggregation.

def get_parameters(model):
    """Extract model weights as a list of numpy arrays."""
    return [val.cpu().numpy() for val in model.state_dict().values()]

def set_parameters(model, parameters):
    """Load aggregated weights back into the model."""
    params_dict = zip(model.state_dict().keys(), parameters)
    state_dict  = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
    model.load_state_dict(state_dict, strict=True)

In [6]:
# FLOWER CLIENT
# ─────────────────────────────────────────────
class TBClient(fl.client.NumPyClient):
    def __init__(self, client_id: int):
        self.client_id   = client_id
        self.model       = TB_CNN().to(device)
        self.train_loader = get_client_loader(client_id)
        self.criterion   = nn.CrossEntropyLoss()
        self.optimizer   = torch.optim.Adam(self.model.parameters(), lr=LR)

    def get_parameters(self, config):
        return get_parameters(self.model)

    def fit(self, parameters, config):
        """Receive global weights → train locally → return updated weights."""
        set_parameters(self.model, parameters)
        self.model.train()

        for epoch in range(LOCAL_EPOCHS):
            total_loss, correct, total = 0.0, 0, 0
            for images, labels in self.train_loader:
                images, labels = images.to(device), labels.to(device)
                self.optimizer.zero_grad()
                outputs = self.model(images)
                loss    = self.criterion(outputs, labels)
                loss.backward()
                self.optimizer.step()

                total_loss += loss.item()
                preds       = outputs.argmax(dim=1)
                correct    += (preds == labels).sum().item()
                total      += labels.size(0)

            acc = correct / total
            print(f"  Client {self.client_id} | Epoch {epoch+1}/{LOCAL_EPOCHS} "
                  f"| Loss: {total_loss/len(self.train_loader):.4f} | Acc: {acc*100:.2f}%")

        return get_parameters(self.model), len(self.train_loader.dataset), {}

    def evaluate(self, parameters, config):
        """Receive global weights → evaluate on local val data → return loss + accuracy."""
        set_parameters(self.model, parameters)
        self.model.eval()

        criterion = nn.CrossEntropyLoss()
        loss_total, correct, total = 0.0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs     = self.model(images)
                loss        = criterion(outputs, labels)
                loss_total += loss.item()
                preds       = outputs.argmax(dim=1)
                correct    += (preds == labels).sum().item()
                total      += labels.size(0)

        avg_loss = loss_total / len(val_loader)
        accuracy = correct / total
        return avg_loss, total, {"accuracy": accuracy}

In [7]:

# ─────────────────────────────────────────────
# 5. FLOWER SERVER STRATEGY
#    Using FedAvg — the standard aggregation strategy
#    FedProx option is commented below (uncomment to use)
# ─────────────────────────────────────────────
# Tracking metrics across rounds
round_accuracies = []
round_losses     = []

def fit_config(server_round: int):
    """Send round number to clients (optional config)."""
    return {"server_round": server_round, "local_epochs": LOCAL_EPOCHS}

def evaluate_config(server_round: int):
    return {"server_round": server_round}

def weighted_average(metrics):
    """Aggregate accuracy from all clients using weighted average."""
    accuracies = [num_examples * m["accuracy"] for num_examples, m in metrics]
    examples   = [num_examples for num_examples, _ in metrics]
    avg_acc    = sum(accuracies) / sum(examples)
    round_accuracies.append(avg_acc)
    print(f"\n  🌐 Global Val Accuracy this round: {avg_acc*100:.2f}%\n")
    return {"accuracy": avg_acc}

strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0,               # Use ALL clients each round
    fraction_evaluate=1.0,
    min_fit_clients=NUM_CLIENTS,
    min_evaluate_clients=NUM_CLIENTS,
    min_available_clients=NUM_CLIENTS,
    on_fit_config_fn=fit_config,
    on_evaluate_config_fn=evaluate_config,
    evaluate_metrics_aggregation_fn=weighted_average,
    initial_parameters=fl.common.ndarrays_to_parameters(
        get_parameters(TB_CNN())    # Initialize with random weights
    ),
)

# ── FedProx (alternative to FedAvg) ───────────────────────
# Uncomment below and comment out FedAvg above to use FedProx.
# FedProx adds a proximal term to penalize client drift.
#
# strategy = fl.server.strategy.FedProx(
#     proximal_mu=0.1,              # Proximal term strength (tune: 0.01–1.0)
#     fraction_fit=1.0,
#     fraction_evaluate=1.0,
#     min_fit_clients=NUM_CLIENTS,
#     min_evaluate_clients=NUM_CLIENTS,
#     min_available_clients=NUM_CLIENTS,
#     on_fit_config_fn=fit_config,
#     evaluate_metrics_aggregation_fn=weighted_average,
#     initial_parameters=fl.common.ndarrays_to_parameters(
#         get_parameters(TB_CNN())
#     ),
# )


In [8]:
def client_fn(cid: str) -> fl.client.Client:
    client_id = int(cid)
    return TBClient(client_id).to_client()


In [1]:
# Resolve protobuf/ray/flwr version conflicts
print("Uninstalling conflicting packages...")
!pip uninstall -y protobuf ray flwr

print("Installing compatible protobuf version...")
!pip install protobuf==6.33.6

print("Installing flwr with simulation dependencies...")
!pip install flwr==1.29.0 "flwr[simulation]"

print("Installing compatible ray version...")
!pip install ray==2.51.1

print("Installation complete. Now attempting to run simulation.")

Uninstalling conflicting packages...
Found existing installation: protobuf 7.34.1
Uninstalling protobuf-7.34.1:
  Successfully uninstalled protobuf-7.34.1
Found existing installation: ray 2.55.1
Uninstalling ray-2.55.1:
  Successfully uninstalled ray-2.55.1
Found existing installation: flwr 1.29.0
Uninstalling flwr-1.29.0:
  Successfully uninstalled flwr-1.29.0
Installing compatible protobuf version...
  Using cached protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
Using cached protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl (323 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.
g

Installing flwr with simulation dependencies...
  Using cached flwr-1.29.0-py3-none-any.whl.metadata (14 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached ray-2.51.1-cp312-cp312-manylinux2014_x86_64.whl.metadata (21 kB)
Using cached flwr-1.29.0-py3-none-any.whl (820 kB)
Using cached ray-2.51.1-cp312-cp312-manylinux2014_x86_64.whl (71.4 MB)
Using cached packaging-25.0-py3-none-any.whl (66 kB)
  Attempting uninstall: packaging
    Found existing installation: packaging 26.1
    Uninstalling packaging-26.1:
      Successfully uninstalled packaging-26.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 

Installing compatible ray version...
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
^C
Installation complete. Now attempting to run simulation.


In [9]:
if __name__ == "__main__":
    print("=" * 55)
    print(" Federated Learning — TB Detection")
    print(f" Clients: {NUM_CLIENTS} | Rounds: {NUM_ROUNDS} | Local Epochs: {LOCAL_EPOCHS}")
    print("=" * 55)

    history = fl.simulation.start_simulation(
        client_fn=client_fn,
        num_clients=NUM_CLIENTS,
        config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
        strategy=strategy,
        client_resources={"num_cpus": 1},   # increase if you have more cores
    )

    # ── Save final aggregated model ────────────────────────
    # After simulation, extract the final global weights
    final_model = TB_CNN()
    final_params = fl.common.parameters_to_ndarrays(
        history.parameters_prime if hasattr(history, 'parameters_prime')
        else fl.common.ndarrays_to_parameters(get_parameters(final_model))
    )
    # Note: Flower simulation doesn't expose final params directly in all versions.
    # Use the save in client as fallback — the model at last round is the best.
    torch.save(final_model.state_dict(), MODEL_SAVE_PATH)
    print(f"\n✅ Federated model saved to {MODEL_SAVE_PATH}")

    # ── Plot federated accuracy across rounds ──────────────
    if round_accuracies:
        plt.figure(figsize=(9, 5))
        plt.plot(range(1, len(round_accuracies) + 1),
                 [a * 100 for a in round_accuracies], 'g-o')
        plt.title('Federated Learning — Global Accuracy per Round (TB)')
        plt.xlabel('Round')
        plt.ylabel('Accuracy (%)')
        plt.grid(True)
        plt.tight_layout()
        plt.savefig('fl_accuracy_curve.png', dpi=150)
        plt.show()
        print("FL accuracy curve saved.")

    print("\nFederated Learning complete.")

 Federated Learning — TB Detection
 Clients: 3 | Rounds: 10 | Local Epochs: 3


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Streaming output truncated to the last 5000 lines.
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 933, in _apply
    module._apply(fn)
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 964, in _apply
    param_applied = fn(param)
                    ^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1367, in convert
    return t.to(
           ^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py", line 424, in _lazy_init
    torch._C._cuda_init()
RuntimeError: No CUDA GPUs are available

The above exception was the direct cause of the following exception:

ray::ClientAppActor.run() (pid=26271, ip=172.28.0.12, actor_id=8641e16893d45ac706f9993c01000000, repr=<flwr.simulation.ray_transport.ray_actor.ClientAppActor object at 0x7d540743a900>)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python

RuntimeError: Parent directory ML/Models does not exist.

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
